# P-2026-11-19-1: Does dropping the commodity columns make the lab's grouped ECI better?

**Question** (from `proposals/2026-11-18.md` #2): the 11-18 run noticed, post hoc, that grouped ECI without *Energy Drilling and Mining* (O4-oriented) has a much higher Spearman with log GDPpc than the 18-column base (0.646–0.716 vs 0.480–0.554). Tonight's run is the pre-registered version. We pre-state the metrics and the decision rule, then score drop-Energy, drop-Energy+Precious Stones and drop-Energy+Precious Stones+Metals against an **income-free** benchmark (the Growth Lab's published ECI) as well as income, and end with a Q10-style 2019→2023 growth pilot.

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`, and every open `claude/nightly-*` branch already has its run merged. Under PROTOCOL §1.4, tonight's question comes from `proposals/`. 11-18 #1 (adopt O4 in `lib/`) would modify the library, which the protocol reserves for questions about `lib/`. #2 is self-contained and decides whether a cheap fix is worth it, so I took it.

- **Date:** 2026-11-19 (America/New_York), nightly run. ID `P-2026-11-19-1`.
- **Data:** `data/cleaned/2019–2023clean.csv` (18 groups, re-keyed on ISO3 via `lib/research.lab_groups_iso3`, EU-28 aggregate dropped); `data/raw/2019–2023_mega.csv` (HS4 ECI reference, plus resource share from HS 26/27/71); `external-data/raw/atlas_hs92_country_year.csv` (Growth Lab published ECI, benchmark only); `external-data/raw/wdi_panel.csv` (GDPpc PPP constant, and constant 2015 US$ for growth). No downloads.
- **Years:** 2019 primary; 2020–2022 as checks; 2023 shown but suspect (D1).
- **Library:** `willbetts/nightly/lib/eci.py`, `research.py` (imported, not modified). O4 orientation is defined in this notebook.
- **Run time:** printed in the last code cell.

## TL;DR

- **No commodity drop passes the pre-stated rule (0 of 3 variants).** Dropping *Energy* raises the Spearman with log GDPpc in every year: **0.480 → 0.664** in 2019, 2019 paired-bootstrap 95% CI on the gain **[0.072, 0.313]**, and +0.150 to +0.184 in 2020–2022. But it **lowers** agreement with the Growth Lab's published ECI in 2020, 2021 and 2022 (−0.040 to −0.057). Its 2019 M1 gain (+0.017) has a CI of [−0.066, 0.104]. V2 (−Energy −Precious) and V3 (−Energy −Precious −Metals) lose M1 in all four years 2019–2022 (−0.031 to −0.133).
- **The income gain comes from resource exporters moving to their income rank, not to their complexity rank.** The 2019 rank gain from V0→V1 has Spearman **0.664** with resource share. Countries with resource share ≥ 0.40 gain **14.2** ranks on average (n = 32), and the rest lose 6.2. For that resource-heavy group, the mean distance to the income rank falls **28.6 → 17.8**, while the distance to the ATLAS ECI rank barely moves (**19.8 → 19.3**). Top gainers: UAE (+63), Norway (+59), Kuwait (+55.5), Brunei (+51.5), Canada (+45).
- **This isn't a linear resource-share artefact either.** Adding resource share to V0 lifts the R² on log GDPpc only from 0.162 to 0.173 (2019), while V1 alone reaches 0.300. Dropping the column changes *which* countries look complex; it doesn't just add a control.
- **The growth pilot is null for every variant**, as in Q10. Over 2019→2023 (n = 105), no variant's coefficient is significant (p = 0.202–0.945), and the commodity-drop coefficients are *negative* (−0.272 to −0.422 per SD). ATLAS ECI also fails to reach significance here (+0.557, p = 0.224).
- **For the lab:** keep the 18 columns. The higher income correlation doesn't track complexity, and it would rewrite reports: UAE goes from 118/120 → 42/120 (2019), Canada from 108 → 54, and the 2019→2022 direction flips for UAE (V1–V3) and Canada (V2–V3). Side result: the lib-default diversity orientation (O1) disagrees with O4 in **14 of 20** dropped/base matrices (2019 V1 margin 0.003), so O1 is unusable for any column-drop variant. All 20 O4 ECIs correlate positively with HS4 ECI.

## Why this matters for the lab

Every country report prints grouped ECI. Q04 and 11-14 showed that the 18-group ECI is a weak income signal (Spearman ≈ 0.5). If deleting one or three commodity columns fixed that, it would be the cheapest possible improvement: no new data and no change to the template. But the agenda warns (§5) about variants that fit income better for the wrong reason. A resource-heavy rich country (Norway, Qatar, Australia) gets low grouped ECI *because* it specialises in Energy. Delete Energy and that penalty is gone, so its ECI tracks its income more closely. That isn't necessarily a better measure of *capabilities*, which is what ECI is meant to capture and what the growth link (agenda §0, link 1) relies on. This run therefore scores each variant against income **and** against an income-free complexity benchmark, and tests the mechanism directly.

## Method (pre-stated before any variant was scored tonight)

**Variants** (all use the lab rules: drop any country with a zero in any *remaining* column, then strict RCA `>`; RCA recomputed on the remaining columns; sorted eigen-decomposition via SVD):
- **V0** base, 18 columns
- **V1** − Energy Drilling and Mining
- **V2** − Energy − Precious Stones
- **V3** − Energy − Precious Stones − Metals

**Orientation:** O4 (11-18): take the PCI from the same singular pair, flip so PCI correlates **negatively** with ubiquity, and give ECI the consistent sign. We also record the diversity rule (O1, the lib default) and whether each O4 ECI correlates positively with repo HS4 ECI (the 11-18 reference).

**Primary metrics** (Spearman, on the **common sample** each year: countries scored by all four variants that also have ATLAS ECI and WDI GDPpc):
- **M1:** Spearman with the Growth Lab's published ECI, same year. Income-free, so not circular.
- **M2:** Spearman with log GDPpc (PPP, constant).

**Decision rule:** a variant is "better" than V0 only if it beats V0 on **both** M1 and M2 in 2019 **and** in at least 2 of 2020–2022. For 2019, a paired country bootstrap (2,000 draws) gives 95% intervals on the M1 and M2 differences against V0.

**Secondary (pre-stated):** (a) Spearman with repo HS4 ECI; (b) mechanism: R² of log GDPpc on ECI alone and on ECI + resource share (HS 26/27/71). If a variant's income gain disappears once resource share is controlled, it's re-encoding resource dependence; (c) rank changes V0→V1 against resource share; (d) the 7 Spring 2026 report countries' ranks; (e) **growth pilot** (Q10 spec): 2019→2023 annualized real GDPpc growth (constant 2015 US$) ~ log GDPpc 2019 + resource share + z(ECI variant), on one common sample, with HC1 SEs.

**Honesty note:** V1's M2 values were already printed by 11-18, so V1 × M2 isn't fresh evidence. M1, the secondary metrics, V2, V3 and the growth pilot are new tonight.

## Setup

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]
REPORT = R.REPORT_COUNTRIES
VARS = {'V0 base': [],
        'V1 -Energy': ['Energy Drilling and Mining'],
        'V2 -Energy -Precious': ['Energy Drilling and Mining', 'Precious Stones'],
        'V3 -Energy -Precious -Metals': ['Energy Drilling and Mining', 'Precious Stones', 'Metals']}
VN = list(VARS)

def eci_o4(X):
    """Lab rules (drop-any-zero, strict RCA), sorted SVD, O4 orientation. Returns z-scored ECI and diagnostics."""
    M = eci.mcp(eci.notebook_filter(X), strict=True)
    Mv = M.to_numpy(float); kc, kp = Mv.sum(1), Mv.sum(0)
    B = Mv / np.sqrt(kc[:, None] * kp[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    k = U[:, 1] / np.sqrt(kc); q = Vt[1] / np.sqrt(kp)       # consistent pair: B v = s u, s > 0
    s4 = -np.sign(np.corrcoef(q, kp)[0, 1])                    # O4: PCI negative with ubiquity
    s1 = np.sign(np.corrcoef(k, kc)[0, 1])                     # O1: ECI positive with diversity (lib default)
    e = pd.Series(k * s4, index=M.index); e = (e - e.mean()) / e.std()
    return e, {'n': len(M), 'cols': M.shape[1], 'O4 sign': int(s4), 'O1 sign': int(s1),
               'O4 margin |r(PCI,ubiq)|': abs(np.corrcoef(q, kp)[0, 1]), 'O1 margin |r(ECI,div)|': abs(np.corrcoef(k, kc)[0, 1]),
               'lambda2': S[1] ** 2, 'lambda3': S[2] ** 2}

G = {y: R.lab_groups_iso3(y, drop_eu=True) for y in YEARS}
H4 = {y: eci.eci_from_exports(R.repo_hs(y, 4)) for y in YEARS}
XH = {y: R.repo_hs(y, 4) for y in YEARS}
RES = {y: R.resource_share(XH[y]) for y in YEARS}
AT = R.atlas_cy()
ATL = {y: AT.query('year == @y').set_index('iso3').eci for y in YEARS}
LNY = {y: np.log(R.gdppc(y, 'ppp')).dropna() for y in YEARS}
E, D = {}, []
for y in YEARS:
    for v, drop in VARS.items():
        E[y, v], d = eci_o4(G[y].drop(columns=drop))
        d.update({'year': y, 'variant': v, 'Spearman w/ HS4 ECI': R.spearman(E[y, v], H4[y])[0]})
        D.append(d)
COMMON = {}
for y in YEARS:
    s = set(ATL[y].index) & set(LNY[y].index)
    for v in VN: s &= set(E[y, v].index)
    COMMON[y] = sorted(s)
print('lib:', Path(eci.__file__).name, Path(R.__file__).name, '| numpy', np.__version__, '| pandas', pd.__version__)
print('Common-sample size by year:', {y: len(COMMON[y]) for y in YEARS})

lib: eci.py research.py | numpy 2.4.6 | pandas 3.1.1
Common-sample size by year: {2019: 118, 2020: 117, 2021: 104, 2022: 118, 2023: 121}


## Results

### A. Sample, orientation and spectra

In [2]:
DD = pd.DataFrame(D).set_index(['year', 'variant'])
print(DD[['n', 'cols', 'O4 sign', 'O1 sign', 'O4 margin |r(PCI,ubiq)|', 'O1 margin |r(ECI,div)|', 'lambda2', 'lambda3', 'Spearman w/ HS4 ECI']].round(3).to_string())
print('\nO1 and O4 disagree in', int((DD['O4 sign'] != DD['O1 sign']).sum()), 'of', len(DD), 'matrices')
print('O4 ECI correlates positively with repo HS4 ECI in', int((DD['Spearman w/ HS4 ECI'] > 0).sum()), 'of', len(DD), 'matrices')
print('Own-sample n identical across variants every year:', all(DD.xs(y)['n'].nunique() == 1 for y in YEARS))

                                     n  cols  O4 sign  O1 sign  O4 margin |r(PCI,ubiq)|  O1 margin |r(ECI,div)|  lambda2  lambda3  Spearman w/ HS4 ECI
year variant                                                                                                                                          
2019 V0 base                       120    18        1        1                    0.675                   0.311    0.329    0.255                0.735
     V1 -Energy                    120    17        1        1                    0.620                   0.003    0.314    0.219                0.736
     V2 -Energy -Precious          120    16        1       -1                    0.640                   0.132    0.312    0.219                0.669
     V3 -Energy -Precious -Metals  120    15        1       -1                    0.640                   0.198    0.341    0.242                0.657
2020 V0 base                       120    18        1        1                    0.697       

**Takeaway (A):** O1 and O4 disagree in 14 of 20 matrices. These are all 12 dropped variants in 2020–2023 plus V2/V3 in 2019, and all four base matrices agree. The O1 margin falls as low as 0.003 (2019 V1), while the O4 margin stays at 0.580–0.697. All 20 O4 ECIs correlate positively with repo HS4 ECI (the 11-18 reference). Dropping columns never changes the filtered sample (n is identical across variants each year), so the comparisons are on the same countries. Agreement with HS4 ECI falls with each column dropped in 2020–2022 (e.g. 2022: 0.773 → 0.718 → 0.674 → 0.669).

### B. Primary metrics M1 (ATLAS ECI) and M2 (log GDPpc) on the common sample

In [3]:
rows = []
for y in YEARS:
    c = COMMON[y]
    for v in VN:
        rows.append({'year': y, 'variant': v, 'n': len(c),
                     'M1 rho ATLAS': stats.spearmanr(E[y, v][c], ATL[y][c])[0],
                     'M2 rho lnGDPpc': stats.spearmanr(E[y, v][c], LNY[y][c])[0]})
P = pd.DataFrame(rows)
for m in ['M1 rho ATLAS', 'M2 rho lnGDPpc']:
    P[m.split()[0] + ' diff vs V0'] = P[m] - P.groupby('year')[m].transform('first')
print(P.round(3).to_string(index=False))
ref = pd.DataFrame([{'year': y, 'rho(ATLAS ECI, lnGDPpc) same sample': stats.spearmanr(ATL[y][COMMON[y]], LNY[y][COMMON[y]])[0]} for y in YEARS])
print('\nReference ceiling:'); print(ref.round(3).to_string(index=False))
P.round(4).to_csv('primary_metrics.csv', index=False)

 year                      variant   n  M1 rho ATLAS  M2 rho lnGDPpc  M1 diff vs V0  M2 diff vs V0
 2019                      V0 base 118         0.663           0.480          0.000          0.000
 2019                   V1 -Energy 118         0.680           0.664          0.017          0.184
 2019         V2 -Energy -Precious 118         0.632           0.642         -0.031          0.161
 2019 V3 -Energy -Precious -Metals 118         0.627           0.631         -0.035          0.151
 2020                      V0 base 117         0.748           0.534          0.000          0.000
 2020                   V1 -Energy 117         0.691           0.716         -0.056          0.182
 2020         V2 -Energy -Precious 117         0.628           0.683         -0.119          0.149
 2020 V3 -Energy -Precious -Metals 117         0.615           0.672         -0.133          0.138
 2021                      V0 base 104         0.751           0.528          0.000          0.000
 2021     

**Takeaway (B):** M2 (income) rises for every variant in every year (+0.132 to +0.184), and M1 (ATLAS) falls in every case except V1 in 2019 (+0.017). Even the best variant's income ρ (0.716, V1 2020) stays below the ATLAS-ECI ceiling on the same sample (0.756–0.830 over 2019–2022).

In [4]:
# Figure: M1 and M2 by year (two panels, one scale each; no dual axis)
COL = dict(zip(VN, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, m, t in zip(axes, ['M1 rho ATLAS', 'M2 rho lnGDPpc'], ['M1: Spearman with ATLAS published ECI', 'M2: Spearman with log GDPpc (PPP)']):
    for v in VN:
        s = P[P.variant == v]
        ax.plot(s.year, s[m], color=COL[v], lw=2, marker='o', ms=6, label=v)
    ax.axvspan(2022.5, 2023.5, color='#e6e6e3', zorder=0); ax.text(2023, 0.47, '2023\nsuspect', ha='center', fontsize=8, color='#52514e')
    ax.set_title(t, fontsize=10, loc='left'); ax.set_xticks(YEARS); ax.grid(axis='y', color='#ececea', lw=0.8)
    for s_ in ['top', 'right']: ax.spines[s_].set_visible(False)
axes[0].set_ylabel('Spearman ρ (common sample)'); axes[0].set_ylim(0.45, 0.8)
axes[1].legend(frameon=False, fontsize=8, loc='lower left', bbox_to_anchor=(0.0, 0.08))
fig.tight_layout(); fig.savefig(FIG / 'primary_metrics.png', dpi=110); plt.close(fig)
print('saved figures/primary_metrics.png')

saved figures/primary_metrics.png


![primary metrics](figures/primary_metrics.png)

**Takeaway (figure):** the two panels move in opposite directions. Dropping commodity columns raises the income correlation but lowers agreement with the published ECI in most years.

### C. Decision rule and 2019 paired bootstrap

In [5]:
rng = np.random.default_rng(20261119)
c = np.array(COMMON[2019]); B = 2000
idx = rng.integers(0, len(c), size=(B, len(c)))
def boot(v, target):
    a = E[2019, v][c].to_numpy(); b = E[2019, 'V0 base'][c].to_numpy(); t = target[c].to_numpy()
    out = np.array([stats.spearmanr(a[i], t[i])[0] - stats.spearmanr(b[i], t[i])[0] for i in idx])
    return np.percentile(out, [2.5, 97.5])
dec = []
for v in VN[1:]:
    s = P[P.variant == v].set_index('year')
    win = (s['M1 diff vs V0'] > 0) & (s['M2 diff vs V0'] > 0)
    lo1, hi1 = boot(v, ATL[2019]); lo2, hi2 = boot(v, LNY[2019])
    dec.append({'variant': v, 'M1 diff 2019': s.loc[2019, 'M1 diff vs V0'], 'M1 95% CI': f'[{lo1:.3f}, {hi1:.3f}]',
                'M2 diff 2019': s.loc[2019, 'M2 diff vs V0'], 'M2 95% CI': f'[{lo2:.3f}, {hi2:.3f}]',
                'wins both 2019': bool(win[2019]), 'wins both, of 2020-22': int(win[[2020, 2021, 2022]].sum()),
                'M1 wins, of 2019-22': int((s.loc[2019:2022, 'M1 diff vs V0'] > 0).sum()),
                'M2 wins, of 2019-22': int((s.loc[2019:2022, 'M2 diff vs V0'] > 0).sum()),
                'BETTER (pre-stated rule)': bool(win[2019] and win[[2020, 2021, 2022]].sum() >= 2)})
DEC = pd.DataFrame(dec); print(DEC.round(3).to_string(index=False))

                     variant  M1 diff 2019       M1 95% CI  M2 diff 2019      M2 95% CI  wins both 2019  wins both, of 2020-22  M1 wins, of 2019-22  M2 wins, of 2019-22  BETTER (pre-stated rule)
                  V1 -Energy         0.017 [-0.066, 0.104]         0.184 [0.072, 0.313]            True                      0                    1                    4                     False
        V2 -Energy -Precious        -0.031 [-0.131, 0.067]         0.161 [0.024, 0.308]           False                      0                    0                    4                     False
V3 -Energy -Precious -Metals        -0.035 [-0.148, 0.079]         0.151 [0.006, 0.302]           False                      0                    0                    4                     False


**Takeaway (C):** **0 of 3 variants is 'better'.** Only V1 wins both metrics in 2019, and none wins both in any of 2020–2022. M2 gains are significant in 2019 (CIs exclude 0 for all three variants), and no M1 difference is distinguishable from 0 in 2019.

### D. Mechanism: is the income gain just resource dependence?

In [6]:
import statsmodels.api as sm
rows = []
for y in YEARS:
    c = [k for k in COMMON[y] if k in RES[y].index]
    for v in VN:
        z = E[y, v][c]; ly = LNY[y][c]; rs = RES[y][c]
        r_alone = sm.OLS(ly, sm.add_constant(z)).fit().rsquared
        r_res = sm.OLS(ly, sm.add_constant(pd.concat([z, rs], axis=1))).fit().rsquared
        rows.append({'year': y, 'variant': v, 'n': len(c), 'R2 lnY~ECI': r_alone, 'R2 lnY~ECI+resource': r_res,
                     'rho(ECI, resource share)': stats.spearmanr(z, rs)[0]})
MEC = pd.DataFrame(rows)
base_res = pd.DataFrame([{'year': y, 'R2 lnY~resource only': sm.OLS(LNY[y][[k for k in COMMON[y] if k in RES[y].index]],
          sm.add_constant(RES[y][[k for k in COMMON[y] if k in RES[y].index]])).fit().rsquared} for y in YEARS])
MEC = MEC.merge(base_res, on='year')
print(MEC.round(3).to_string(index=False))

 year                      variant   n  R2 lnY~ECI  R2 lnY~ECI+resource  rho(ECI, resource share)  R2 lnY~resource only
 2019                      V0 base 105       0.162                0.173                    -0.620                 0.028
 2019                   V1 -Energy 105       0.300                0.302                    -0.217                 0.028
 2019         V2 -Energy -Precious 105       0.290                0.309                    -0.029                 0.028
 2019 V3 -Energy -Precious -Metals 105       0.231                0.250                    -0.032                 0.028
 2020                      V0 base 105       0.216                0.235                    -0.688                 0.047
 2020                   V1 -Energy 105       0.332                0.341                    -0.249                 0.047
 2020         V2 -Energy -Precious 105       0.281                0.311                    -0.099                 0.047
 2020 V3 -Energy -Precious -Metals 105  

**Takeaway (D):** base grouped ECI is strongly anti-correlated with resource share (ρ −0.605 to −0.688), while V2/V3 are nearly orthogonal to it (−0.015 to −0.099). Resource share alone explains little of log GDPpc (R² 0.010–0.047), and adding it to V0 closes little of the gap to V1 (2019: 0.173 vs 0.300). So the pre-stated 're-encoding' test is negative. The gain isn't a linear resource-share effect; the variants re-rank countries (D2).

In [7]:
# D2: who moves V0 -> V1 in 2019, and is the move explained by resource share?
y = 2019; c = [k for k in COMMON[y] if k in RES[y].index]
rk0 = E[y, 'V0 base'][c].rank(ascending=False); rk1 = E[y, 'V1 -Energy'][c].rank(ascending=False)
MV = pd.DataFrame({'rank V0': rk0, 'rank V1': rk1, 'gain (V0-V1)': rk0 - rk1, 'resource share': RES[y][c],
                   'rank ATLAS': ATL[y][c].rank(ascending=False), 'rank lnGDPpc': LNY[y][c].rank(ascending=False)}).astype(float)
print('Spearman(rank gain V0->V1, resource share) = %.3f, n = %d' % (stats.spearmanr(MV['gain (V0-V1)'], MV['resource share'])[0], len(MV)))
hi = MV['resource share'] >= 0.4
print('Mean rank gain: resource share >= 0.40: %.1f (n=%d); others: %.1f (n=%d)' % (MV.loc[hi, 'gain (V0-V1)'].mean(), hi.sum(), MV.loc[~hi, 'gain (V0-V1)'].mean(), (~hi).sum()))
print('\nTop 10 gainers:'); print(MV.sort_values('gain (V0-V1)', ascending=False).head(10).round(2).to_string())
print('\nTop 5 losers:'); print(MV.sort_values('gain (V0-V1)').head(5).round(2).to_string())

Spearman(rank gain V0->V1, resource share) = 0.664, n = 105
Mean rank gain: resource share >= 0.40: 14.2 (n=32); others: -6.2 (n=73)

Top 10 gainers:
     rank V0  rank V1  gain (V0-V1)  resource share  rank ATLAS  rank lnGDPpc
ARE    103.0     40.0          63.0            0.67        42.0           7.0
NOR     91.0     32.0          59.0            0.59        30.0           4.0
KWT     59.5      4.0          55.5            0.92        60.0          25.0
BRN     59.5      8.0          51.5            0.91        49.0           6.0
CAN     95.0     50.0          45.0            0.30        33.0          19.0
AUS     97.0     59.0          38.0            0.69        63.0          18.0
TTO     51.0     14.0          37.0            0.42        70.0          46.0
COL    101.0     68.0          33.0            0.60        65.0          65.0
AZE    102.0     70.0          32.0            0.92        92.0          59.0
BHR     69.0     45.0          24.0            0.56        46.0       

**Takeaway (D2):** the 2019 rank gain V0→V1 has Spearman 0.664 with resource share. The top gainers are rich hydrocarbon exporters (UAE, Norway, Kuwait, Brunei) plus Canada and Australia. The top losers are low-resource manufacturers (Moldova, Cambodia, Pakistan, Portugal, El Salvador), which fall 31–38 ranks.

In [8]:
# D3: on the resource-heavy group (share >= 0.40), which ranking does V1 move toward?
g = MV[hi]
print('Resource-heavy countries (n=%d): mean |rank V0 - rank ATLAS| = %.1f, |rank V1 - rank ATLAS| = %.1f' % (len(g), (g['rank V0'] - g['rank ATLAS']).abs().mean(), (g['rank V1'] - g['rank ATLAS']).abs().mean()))
print('                              mean |rank V0 - rank lnY|   = %.1f, |rank V1 - rank lnY|   = %.1f' % ((g['rank V0'] - g['rank lnGDPpc']).abs().mean(), (g['rank V1'] - g['rank lnGDPpc']).abs().mean()))
o = MV[~hi]
print('Other countries (n=%d):         mean |rank V0 - rank ATLAS| = %.1f, |rank V1 - rank ATLAS| = %.1f' % (len(o), (o['rank V0'] - o['rank ATLAS']).abs().mean(), (o['rank V1'] - o['rank ATLAS']).abs().mean()))

Resource-heavy countries (n=32): mean |rank V0 - rank ATLAS| = 19.8, |rank V1 - rank ATLAS| = 19.3
                              mean |rank V0 - rank lnY|   = 28.6, |rank V1 - rank lnY|   = 17.8
Other countries (n=73):         mean |rank V0 - rank ATLAS| = 17.5, |rank V1 - rank ATLAS| = 17.1


**Takeaway (D3):** for the 32 resource-heavy countries, V1 closes **10.8** ranks of distance to income rank (28.6 → 17.8) but only **0.5** to the ATLAS rank (19.8 → 19.3). The others barely move relative to ATLAS (17.5 → 17.1). Dropping Energy makes grouped ECI more of an income proxy without making it more like the field's complexity measure.

### E. Growth pilot (Q10 spec), 2019 → 2023

In [9]:
y = 2019
df = pd.DataFrame({'growth': R.growth(2019, 2023, 'const'), 'lny0': np.log(R.gdppc(2019, 'const')), 'resource': RES[y]})
for v in VN:
    df[v] = (E[y, v] - E[y, v].mean()) / E[y, v].std()
df['ATLAS ECI'] = (ATL[y] - ATL[y].mean()) / ATL[y].std()
core = df.dropna(subset=['growth', 'lny0', 'resource'] + VN + ['ATLAS ECI'])
base = R.ols(core, 'growth', ['lny0', 'resource'])
rows = [{'model': 'baseline: lnGDPpc + resource', 'n': base['n'], 'R2': base['r2']}]
for v in VN + ['ATLAS ECI']:
    m = R.ols(core, 'growth', ['lny0', 'resource', v])
    rows.append({'model': '+ ' + v, 'n': m['n'], 'R2': m['r2'], 'dR2': m['r2'] - base['r2'],
                 'coef per SD': m['coef'][v], 'se (HC1)': m['se'][v], 'p': m['p'][v]})
GR = pd.DataFrame(rows); print(GR.round(4).to_string(index=False))
print('growth mean %.2f, sd %.2f %%/yr' % (core.growth.mean(), core.growth.std()))

                         model   n    R2   dR2  coef per SD  se (HC1)     p
  baseline: lnGDPpc + resource 105 0.003   NaN          NaN       NaN   NaN
                     + V0 base 105 0.003 0.000       -0.019     0.271 0.945
                  + V1 -Energy 105 0.009 0.006       -0.348     0.336 0.300
        + V2 -Energy -Precious 105 0.011 0.009       -0.422     0.330 0.202
+ V3 -Energy -Precious -Metals 105 0.007 0.004       -0.272     0.266 0.307
                   + ATLAS ECI 105 0.007 0.004        0.557     0.458 0.224
growth mean 1.27, sd 3.70 %/yr


**Takeaway (E):** null across the board (baseline R² 0.003, every ΔR² ≤ 0.009, every p ≥ 0.202). The commodity-drop coefficients are negative (−0.272 to −0.422 per SD), the opposite of the sign complexity theory predicts, and ATLAS ECI is positive but not significant (0.557, p = 0.224). Over 2019→2023 this pilot can't separate the variants, and nothing here argues for a drop.

### F. Spring 2026 report countries

In [10]:
rows = []
for y in [2019, 2022]:
    for iso, name in REPORT.items():
        r = {'year': y, 'country': name}
        for v in VN:
            e = E[y, v]
            r[v] = f"{int(e.rank(ascending=False)[iso])}/{len(e)}" if iso in e.index else 'n/a'
        rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
print('\n2019->2022 direction of z-scored ECI (sign of change):')
rows = []
for iso, name in REPORT.items():
    rows.append({'country': name, **{v: ('n/a' if iso not in E[2019, v].index or iso not in E[2022, v].index else
                 ('up' if E[2022, v][iso] > E[2019, v][iso] else 'down')) for v in VN}})
print(pd.DataFrame(rows).to_string(index=False))

 year   country V0 base V1 -Energy V2 -Energy -Precious V3 -Energy -Precious -Metals
 2019    Canada 108/120     54/120               48/120                       42/120
 2019   Germany   1/120      2/120                3/120                        5/120
 2019    Poland  36/120     50/120               60/120                       57/120
 2019 Singapore  28/120     13/120                7/120                        8/120
 2019       UAE 118/120     42/120               29/120                       43/120
 2019        UK  39/120     35/120               12/120                       18/120
 2019   Vietnam  34/120     48/120               58/120                       56/120
 2022    Canada  80/120     48/120               51/120                       43/120
 2022   Germany   8/120     22/120               26/120                       27/120
 2022    Poland  41/120     57/120               68/120                       68/120
 2022 Singapore   9/120      9/120                4/120          

**Takeaway (F):** the variants would rewrite resource-exporter reports. In 2019 the UAE goes from 118/120 (V0) to 42/29/43 and Canada from 108 to 54/48/42, while Poland and Vietnam fall (36 → 50–60 and 34 → 48–58). The 2019→2022 direction flips for the UAE (up → down under V1–V3) and Canada (up → down under V2–V3). Germany, Poland, Singapore, the UK and Vietnam keep their direction.

## Caveats

- **Small, overlapping samples.** 104–121 countries per year, and the years aren't independent (same countries, persistent ECI). The bootstrap covers 2019 only.
- **ATLAS ECI isn't ground truth.** It uses mirror-reconciled HS4 trade, a population and export floor, and (from 2026) continuous RCA. It's a benchmark for "complexity as the field measures it", not for capabilities as such. Still, it's the only income-free check available in the repo.
- **Nominal values**, and the lab's grouping keeps D4's mapping (84 only in the aggregate, 85 in Transportation). Dropping columns doesn't fix D4.
- **2023 is suspect (D1)**, so it's shown but excluded from the decision rule.
- **Resource share** is the crude HS 26/27/71 definition (Maitree's findings note that it misses refined copper, ch. 74).
- **Growth pilot:** a 4-year window that includes COVID. Q10 found no predictor significant here, even ATLAS ECI. A null result says little about the 10-year link.
- V1 × M2 was seen before pre-registration (11-18), as noted in the Method.

## Helpful to know

- **Never orient a column-drop variant by diversity.** O1 disagreed with O4 in 14 of 20 matrices tonight, and wherever they disagreed, O1 was the one that failed the HS4 sign check (O4 passed 20/20). The 11-18 run saw the same thing for single drops. `eci_o4()` in this notebook (about 10 lines, consistent SVD pair) can be copied as-is into `lib/eci.py` when 11-18 proposal #1 is done.
- Dropping columns under the drop-any-zero rule **never changed the sample** in 2019–2023. The countries the filter removes have zeros in other columns too (not only in Energy, Precious Stones or Metals).
- **"Higher Spearman with income" isn't a sufficient criterion for an ECI variant.** On grouped data it can rise by +0.18 while agreement with published ECI and with HS4 ECI falls. Future variant tests should always carry an income-free benchmark (ATLAS ECI and/or repo HS4 ECI) alongside income.
- The same-sample ceiling for grouped variants is ATLAS ECI's own income ρ: 0.756–0.830 in 2019–2022.

## New research avenues (also in `proposals/2026-11-19.md`)

1. **Resource-exporter correction that keeps the column:** instead of dropping Energy, score Atlas-style "ECI + resource controls". Regress grouped ECI on resource share and report the residual, or reweight RCA by excluding Energy only from the *denominator* (shares of non-resource exports). Pre-state M1/M2 as tonight. One night.
2. **Same test at HS4:** drop chapters 26/27/71 from repo HS4 ECI and score M1/M2. If HS4 also gains income ρ and loses ATLAS ρ, the effect is general. If HS4 loses neither, the grouped result is a coarse-grid artefact. One night.
3. **Adopt O4 in `lib/eci.py`** (11-18 #1, still open). Tonight adds 14 more O1/O4 disagreements to the evidence.
4. **Leave-one-resource-exporter-out on M1:** check whether V1's 2020–2022 M1 losses come from a handful of Gulf countries (ARE, KWT, BRN, BHR) whose ATLAS ECI is mid-table. Low priority.

## Data issues / data needed

None new tonight. Everything used is already in the repo or in `external-data/raw/`.

In [11]:
print('Run time: %.1f s' % (time.time() - T0))

Run time: 8.9 s
